# Transfer Learning CIFAR10

* Train a simple convnet on the CIFAR dataset the first 5 output classes [0..4].
* Freeze convolutional layers and fine-tune dense layers for the last 5 ouput classes [5..9].


In [1]:
import tensorflow as tf
import keras

import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.utils import np_utils
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from keras.constraints import maxnorm

#Downloading the Dataset.
from keras.datasets import cifar10

Using TensorFlow backend.


### 1. Import CIFAR10 data and create 2 datasets with one dataset having classes from 0 to 4 and other having classes from 5 to 9 

In [2]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

170500096/170498071 [==============================] - 9s 0us/step


In [0]:
X_train_0_to_4 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
X_train_5_to_9 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

X_test_0_to_4 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
X_test_5_to_9 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [0]:
y_train_0_to_4 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
y_train_5_to_9 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

y_test_0_to_4 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
y_test_5_to_9 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [5]:
np.unique(y_test_0_to_4)

array([0, 1, 2, 3, 4])

In [6]:
np.unique(y_test_5_to_9)

array([5, 6, 7, 8, 9])

### 2. Use One-hot encoding to divide y_train and y_test into required no of output classes

In [0]:
num_class_0_to_4 = len(np.unique(y_train_0_to_4))

y_train_0_to_4 = tf.keras.utils.to_categorical(y_train_0_to_4, num_classes=num_class_0_to_4)
y_test_0_to_4 = tf.keras.utils.to_categorical(y_test_0_to_4, num_classes=num_class_0_to_4)

### 3. Build a sequential neural network model which can classify the classes 0 to 4 of CIFAR10 dataset with at least 80% accuracy on test data

In [0]:
# normalize inputs from 0-255 to 0.0-1.0
X_train_0_to_4 = X_train_0_to_4.astype('float32')
X_test_0_to_4 = X_test_0_to_4.astype('float32')

X_train_0_to_4 = X_train_0_to_4 / 255.0
X_test_0_to_4 = X_test_0_to_4 / 255.0

In [9]:
# initialize model
model = Sequential()

# add input layer with input_size(32,32,3)
model.add(Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu', padding='same'))

# create Conv 2D layers , along with DropoUts and MaxPooling to avoid overfit
model.add(Dropout(0.2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
# Flatten output
model.add(Flatten())
model.add(Dropout(0.2))
# add Dense layers
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.2))

# add Output layer
model.add(Dense(num_class_0_to_4, activation='softmax'))

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [10]:
# Compile model
from keras.optimizers import SGD
epochs = 25
lrate = 0.01
decay = lrate/epochs
sgd = tf.keras.optimizers.SGD(lr=lrate, momentum=0.9, decay=decay, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 32, 32, 32)        896       
_________________________________________________________________
dropout (Dropout)            (None, 32, 32, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 32, 32, 32)        9248      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 16, 16, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 16, 16, 64)        18496     
_________________________________________________________________
dropout_1 (Dropout)          (None, 16, 16, 64)        0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 16, 16, 64)        3

In [11]:
# Fit the model
model.fit(X_train_0_to_4, y_train_0_to_4, validation_data=(X_test_0_to_4, y_test_0_to_4), epochs=epochs, batch_size=64)


Train on 25000 samples, validate on 5000 samples
Epoch 1/25
25000/25000 [==============================] - 8s 340us/sample - loss: 1.4146 - acc: 0.3866 - val_loss: 1.2341 - val_acc: 0.4644
Epoch 2/25
25000/25000 [==============================] - 4s 168us/sample - loss: 1.0946 - acc: 0.5470 - val_loss: 0.9814 - val_acc: 0.6106
Epoch 3/25
25000/25000 [==============================] - 4s 168us/sample - loss: 0.9407 - acc: 0.6207 - val_loss: 1.0050 - val_acc: 0.5836
Epoch 4/25
25000/25000 [==============================] - 4s 166us/sample - loss: 0.8598 - acc: 0.6543 - val_loss: 0.7972 - val_acc: 0.6766
Epoch 5/25
25000/25000 [==============================] - 4s 167us/sample - loss: 0.7932 - acc: 0.6893 - val_loss: 0.7732 - val_acc: 0.7012
Epoch 6/25
25000/25000 [==============================] - 4s 168us/sample - loss: 0.7408 - acc: 0.7117 - val_loss: 0.7253 - val_acc: 0.7152
Epoch 7/25
25000/25000 [==============================] - 4s 168us/sample - loss: 0.7051 - acc: 0.7295 - val_lo

In [12]:
score = model.evaluate(X_test_0_to_4, y_test_0_to_4)
print('\n', 'Test accuracy for Model :', score[1]*100)

5000/5000 [==============================] - 0s 92us/sample - loss: 0.5207 - acc: 0.8148

 Test accuracy for Model : 81.48000240325928


### 4. In the model which was built above (for classification of classes 0-4 in CIFAR10), make only the dense layers to be trainable and conv layers to be non-trainable

In [0]:
#Set pre-trained model layers to not trainable
for layer in model.layers[:-5]:
    layer.trainable = False

In [14]:
for layer in model.layers:
    print(layer, layer.trainable)

<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdffe5f98> False
<tensorflow.python.keras.layers.core.Dropout object at 0x7fdfdfff8198> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdfff8cc0> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x7fdfdfff8898> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdf64f550> False
<tensorflow.python.keras.layers.core.Dropout object at 0x7fdfdffc3470> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdfff8a58> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x7fdfdf64f7f0> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdf72b470> False
<tensorflow.python.keras.layers.core.Dropout object at 0x7fdfdf72b748> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x7fdfdf6e3f28> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x7fdfdf6cf3c8> False
<ten

### 5. Utilize the the model trained on CIFAR 10 (classes 0 to 4) to classify the classes 5 to 9 of CIFAR 10  (Use Transfer Learning) <br>
Achieve an accuracy of more than 85% on test data

In [0]:
# pre process labels
y_train_5_to_9 = y_train_5_to_9 - 5
y_test_5_to_9 = y_test_5_to_9 - 5

In [0]:
# one hot encoding
num_class_5_to_9 = len(np.unique(y_train_5_to_9))

y_train_5_to_9 = tf.keras.utils.to_categorical(y_train_5_to_9, num_classes=num_class_5_to_9)
y_test_5_to_9 = tf.keras.utils.to_categorical(y_test_5_to_9, num_classes=num_class_5_to_9)

In [0]:
# normalizze input
X_train_5_to_9 = X_train_5_to_9.astype("float32")/255.0
X_test_5_to_9 = X_test_5_to_9.astype("float32")/255.0

In [18]:
# fit the model by retraining only dense layers 

epochs=15
batch_size = 64

model.fit(X_train_5_to_9, y_train_5_to_9, validation_data=(X_test_5_to_9, y_test_5_to_9), batch_size=batch_size, epochs=epochs)

Train on 25000 samples, validate on 5000 samples
Epoch 1/15
25000/25000 [==============================] - 4s 169us/sample - loss: 0.9465 - acc: 0.6448 - val_loss: 0.6188 - val_acc: 0.7754
Epoch 2/15
25000/25000 [==============================] - 4s 168us/sample - loss: 0.6138 - acc: 0.7703 - val_loss: 0.5159 - val_acc: 0.8120
Epoch 3/15
25000/25000 [==============================] - 4s 168us/sample - loss: 0.5268 - acc: 0.8030 - val_loss: 0.4638 - val_acc: 0.8306
Epoch 4/15
25000/25000 [==============================] - 4s 168us/sample - loss: 0.4789 - acc: 0.8215 - val_loss: 0.4220 - val_acc: 0.8472
Epoch 5/15
25000/25000 [==============================] - 4s 168us/sample - loss: 0.4407 - acc: 0.8388 - val_loss: 0.4153 - val_acc: 0.8494
Epoch 6/15
25000/25000 [==============================] - 4s 167us/sample - loss: 0.4094 - acc: 0.8484 - val_loss: 0.3980 - val_acc: 0.8564
Epoch 7/15
25000/25000 [==============================] - 4s 168us/sample - loss: 0.3867 - acc: 0.8549 - val_lo

In [19]:
score = model.evaluate(X_test_5_to_9, y_test_5_to_9)
print('\n', 'Test accuracy for Model with Transfer Learning :', score[1]*100)

5000/5000 [==============================] - 1s 107us/sample - loss: 0.3517 - acc: 0.8770

 Test accuracy for Model with Transfer Learning : 87.69999742507935


# Text classification using TF-IDF

### 6. Load the dataset from sklearn.datasets

In [0]:
from sklearn.datasets import fetch_20newsgroups

In [0]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

### 7. Training data

In [22]:
twenty_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

### 8. Test data

In [0]:
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

###  a.  You can access the values for the target variable using .target attribute 
###  b. You can access the name of the class in the target variable with .target_names


In [24]:
twenty_train.target

array([1, 1, 3, ..., 2, 2, 2])

In [33]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [25]:
twenty_train.data[0:5]

['From: sd345@city.ac.uk (Michael Collier)\nSubject: Converting images to HP LaserJet III?\nNntp-Posting-Host: hampton\nOrganization: The City University\nLines: 14\n\nDoes anyone know of a good way (standard PC application/PD utility) to\nconvert tif/img/tga files into LaserJet III format.  We would also like to\ndo the same, converting to HPGL (HP plotter) files.\n\nPlease email any response.\n\nIs this the correct group?\n\nThanks in advance.  Michael.\n-- \nMichael Collier (Programmer)                 The Computer Unit,\nEmail: M.P.Collier@uk.ac.city                The City University,\nTel: 071 477-8000 x3769                      London,\nFax: 071 477-8565                            EC1V 0HB.\n',
 "From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\nSubject: help: Splitting a trimming region along a mesh \nOrganization: University Of Kentucky, Dept. of Math Sciences\nLines: 28\n\n\n\n\tHi,\n\n\tI have a problem, I hope some of the 'gurus' can help me solve.\n\n\tBackground of the probl

### 9.  Now with dependent and independent data available for both train and test datasets, using TfidfVectorizer fit and transform the training data and test data and get the tfidf features for both

In [0]:
# use TfIDFVectorizer to create document-term matrices 
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()

In [0]:
twenty_train_dtm = vect.fit_transform(twenty_train.data)

In [0]:
twenty_test_dtm = vect.transform(twenty_test.data)

### 10. Use logisticRegression with tfidf features as input and targets as output and train the model and report the train and test accuracy score

In [0]:
# import and instantiate a logistic regression model
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression()

In [30]:
# train the model using X_train_dtm
logreg.fit(twenty_train_dtm, twenty_train.target)

/usr/local/lib/python3.6/dist-packages/sklearn/linear_model/logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/linear_model/logistic.py:469: FutureWarning: Default multi_class will be changed to 'auto' in 0.22. Specify the multi_class option to silence this warning.
  "this warning.", FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=None, solver='warn', tol=0.0001, verbose=0,
                   warm_start=False)

In [0]:
# make class predictions for X_test_dtm
y_pred_class_log = logreg.predict(twenty_test_dtm)

In [32]:
# calculate accuracy
from sklearn import metrics

metrics.accuracy_score(twenty_test.target, y_pred_class_log)

0.8868175765645806